# Denoising Diffusion Implicit Models

**Author:** [András Béres](https://www.linkedin.com/in/andras-beres-789190210)<br>
**Date created:** 2022/06/24<br>
**Last modified:** 2022/06/24<br>
**Description:** Generating images of flowers with denoising diffusion implicit models.

**Modificado:** Valero & Jordi

Este es:
https://keras.io/examples/generative/ddim/

Mira también:

https://keras.io/examples/generative/ddpm/

https://keras.io/examples/generative/finetune_stable_diffusion/


## Introducción

### Qúe son los modelos de difusión?

La difusión se refiere al proceso de convertir una señal estructurada (una imagen) en ruido paso a paso. Simulando difusión podemos generar imágenes ruidosas desde imágenes límpias y entrenar una red para que las limpie. Luego con esta red podemos revertir el proceso, difusión inversa, con lo cual generamos imágenes a partir de ruido.

![diffusion process gif](https://i.imgur.com/dipPOfa.gif)

En resumen: **los modelos de difusión se entrenan primero para limpiar el ruido de imágenes, lo que permite generar imágenes limpiado ruido paso a paso.**

Este notebook implementa una versión continua de [Denoising Diffusion Implicit Models (DDIMs)](https://arxiv.org/abs/2010.02502) usando muestreo determinista.

## Setup

In [ ]:
# import os

import math
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds

import keras
from keras import layers
from tensorflow.keras import ops

## Hiperparámetros

In [ ]:
# data
dataset_name = "oxford_flowers102"
dataset_repetitions = 5
num_epochs = 1  # train for at least 50 epochs for good results
image_size = 64

plot_diffusion_steps = 20

# sampling
min_signal_rate = 0.02
max_signal_rate = 0.95

# architecture
embedding_dims = 32
embedding_max_frequency = 1000.0
widths = [32, 64, 96, 128]
block_depth = 2

# optimization
batch_size = 64
ema = 0.999 # track exponential moving averages weights
learning_rate = 1e-3
weight_decay = 1e-4

## Datos

Usaremos [Oxford Flowers 102](https://www.tensorflow.org/datasets/catalog/oxford_flowers102) para generer imágenes de flores. Este dataset contiene unas  8,000 imágenes.

Los conjuntos de entrenamiento y validación del dataset original están desbalanceados y la mayoría de imágenes están en el conjunto de test. Por ello crearemos una nueva partición (80% entrenamiento, 20% validación) usando la
[Tensorflow Datasets slicing API](https://www.tensorflow.org/datasets/splits):

- Para el entrenamiento se usa `train[:80%]+validation[:80%]+test[:80%]`, es decir, se toma el 80% (del principio **hasta**, `:80%`) de las imágenes que haya de train, de validation y de test.
- Para validación se usa `train[80%:]+validation[80%:]+test[:80%:]`. Ojo a donde están los `:`, aquí se toma **desde** el 80% en adelante, es decir, el 20% restante.

Además se recortan y centran las imágenes, y se repite el mismo dataset varias veces (para tener más muestras).

In [ ]:

def preprocess_image(data):
    # center crop image
    height = keras.ops.shape(data["image"])[0]
    width = keras.ops.shape(data["image"])[1]
    crop_size = ops.minimum(height, width)
    image = tf.image.crop_to_bounding_box(
        data["image"],
        (height - crop_size) // 2,
        (width - crop_size) // 2,
        crop_size,
        crop_size,
    )

    # resize and clip
    # for image downsampling it is important to turn on antialiasing
    image = tf.image.resize(image, size=[image_size, image_size], antialias=True)
    return ops.clip(image / 255.0, 0.0, 1.0)


def prepare_dataset(split):
    return (
        tfds.load(dataset_name, split=split, shuffle_files=True)
        .map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
        .cache()
        .repeat(dataset_repetitions)
        .shuffle(10 * batch_size)
        .batch(batch_size, drop_remainder=True)
        .prefetch(buffer_size=tf.data.AUTOTUNE)
    )


# load dataset
train_dataset = prepare_dataset("train[:80%]+validation[:80%]+test[:80%]")
val_dataset = prepare_dataset("train[80%:]+validation[80%:]+test[80%:]")

## Arquitectura de la red

Usaremos una arquitectura [U-Net](https://arxiv.org/abs/1505.04597) con las mismas entradas y salidas. Básicamente se trata de un autoencoder que como siempre aplica progresivamente un *downsampling* y luego un *upsampling*. La diferencia es que se añaden conexiones directas entre capas con la misma resolución. Estas conexiones ayudan a calcular los gradientes y evitan el cuello de botella de los [autoencoders](https://www.deeplearningbook.org/contents/autoencoders.html). Se pueden ver los modelos de difusión como autoencoders de limpieza de ruido, [denoising autoencoders](https://benanne.github.io/2022/01/31/diffusion.html).

La red toma dos entradas, las imágenes ruidosas y las varianzas del ruido. Estas varianzas son las que se usan para limpiar la imagen. Las varianzas se transforman usando *sinusoidal embeddings*, parecidos a las codificaciones posicionales que se usan en [transformers](https://arxiv.org/abs/1706.03762) y
[NeRF](https://arxiv.org/abs/2003.08934). Básicamente esto es como una representación en el dominio de la frecuencia. Se usa esta transformación porque de esta forma la red es [muy sensible](https://arxiv.org/abs/2006.10739) al nivel del ruido, lo cual es crucial para un buen rendimiento.

Más aspectos a tener en cuenta:

* La red se construye utilizando la [API funcional de Keras](https://keras.io/guides/functional_api/).

* Los [modelos de difusión](https://arxiv.org/abs/2006.11239) usan un índice temporal (*timestep*) para el proceso de difusión. En esta implementación se usa una función que depende de la varianza del ruido (es una estrategiia [score-based](https://arxiv.org/abs/2206.00364)), que la hace independiente del número total de *timesteps* utilizado en el entrenamiento. Esto permite cambiar el *samplig schedule* en inferencia sin tener que reentrenar la red.

* Los [modelos de difusión](https://arxiv.org/abs/2006.11239) introducen el embedding en cada bloque de convolución. Aquí solo se mete al principio. Esto se hace para que el modelo sea más sencillo, y además no afecta casi al rendimiento porque las conexiones directas de la U-Net ya propagan correctamente la información.

* Se desactivan los parámetros de centrado y escalado de las capas *batch normalization* porque las siguientes capas de convolución las hacen redundantes => menos parámetros que entrenar!

* El *kernel* de la última capa de convolución se inicializa a cero para que la red prediga ceros al principio, que es el valor promedio. Esto mejora el entrenamiento el inicio y hace que el MSE valga justo 1 al inicio.

In [ ]:
@keras.saving.register_keras_serializable()
def sinusoidal_embedding(x):
    embedding_min_frequency = 1.0
    frequencies = ops.exp(
        ops.linspace(
            ops.log(embedding_min_frequency),
            ops.log(embedding_max_frequency),
            embedding_dims // 2,
        )
    )
    angular_speeds = ops.cast(2.0 * math.pi * frequencies, "float32")
    embeddings = ops.concatenate(
        [ops.sin(angular_speeds * x), ops.cos(angular_speeds * x)], axis=3
    )
    return embeddings


def ResidualBlock(width):
    def apply(x):
        input_width = x.shape[3]
        if input_width == width:
            residual = x
        else:
            residual = layers.Conv2D(width, kernel_size=1)(x)
        x = layers.BatchNormalization(center=False, scale=False)(x)
        x = layers.Conv2D(width, kernel_size=3, padding="same", activation="swish")(x)
        x = layers.Conv2D(width, kernel_size=3, padding="same")(x)
        x = layers.Add()([x, residual])
        return x

    return apply


def DownBlock(width, block_depth):
    def apply(x):
        x, skips = x
        for _ in range(block_depth):
            x = ResidualBlock(width)(x)
            skips.append(x)
        x = layers.AveragePooling2D(pool_size=2)(x)
        return x

    return apply


def UpBlock(width, block_depth):
    def apply(x):
        x, skips = x
        x = layers.UpSampling2D(size=2, interpolation="bilinear")(x)
        for _ in range(block_depth):
            x = layers.Concatenate()([x, skips.pop()])
            x = ResidualBlock(width)(x)
        return x

    return apply


def get_network(image_size, widths, block_depth):
    noisy_images = keras.Input(shape=(image_size, image_size, 3))
    noise_variances = keras.Input(shape=(1, 1, 1))

    e = layers.Lambda(sinusoidal_embedding, output_shape=(1, 1, 32))(noise_variances)
    e = layers.UpSampling2D(size=image_size, interpolation="nearest")(e)

    x = layers.Conv2D(widths[0], kernel_size=1)(noisy_images)
    x = layers.Concatenate()([x, e])

    skips = []
    for width in widths[:-1]:
        x = DownBlock(width, block_depth)([x, skips])

    for _ in range(block_depth):
        x = ResidualBlock(widths[-1])(x)

    for width in reversed(widths[:-1]):
        x = UpBlock(width, block_depth)([x, skips])

    x = layers.Conv2D(3, kernel_size=1, kernel_initializer="zeros")(x)

    return keras.Model([noisy_images, noise_variances], x, name="residual_unet")

Para echarle un vistazo a la arquitectura de la red ...

In [ ]:
tf.keras.utils.plot_model(get_network(image_size, widths, block_depth), show_shapes=True, dpi=64)

## Diffusion model

### Diffusion scheduler (planificador de difusión)

El proceso de difusión de produce en pasos temporales, normalmente empiezan en t=0 y acaban en t=1. Esta variable es el tiempo de difusión (*diffusion time*), y puede ser discreta (lo más habitual) o continua (se usan en modelos de difusión basados en "puntuación", *scored-based models*). Aquí se usa esta última porque como se ha dicho antes de esta forma se puede cambiar el número de pasos de muestreo al hacer inferencia sin tener que volver a re-entrenar la red.

Se necesita una función que en cada instante temporal indique el nivel de señal y de ruido de la imagen ruidosa. Esta función se implementa en el plafinicador de difusión, o *diffusion scheduler*.

El planificador tiene que dar dos salidas, el ratio de ruido y el ratio de señal (las variables `noise_rate` y `signal_rate` en el código). En el artículo DDIM estas se corresponden con $\sqrt{(1-\alpha)}$ y $\sqrt{\alpha}$. La imagen ruidosa se genera combinando la imagen limpia y el ruido pesados con
estos niveles de señal y ruido.

Puesto que tanto el ruido Gaussiano como las imágenes normalizadas tienen media cero y varianza unidad, los ratios de ruido y señal que proporciona el planificador se pueden intepretar como la desviación estándar de los píxeles de la imagen ruidosa, mientras que sus valores al cuadrado serían sus varianzas (es decir, su potencia desde el punto de vista del procesado de la señal). Se fuerza a que la suma al cuadrado de los ratios de ruido y señal sea 1, lo que implica que la señal ruidosa tendrá siempre varianza unidad, igual que sus componentes por separado.

Usaremos una versión simplificada y continua de un planificador coseno,
[cosine scheduler (Section 3.2)](https://arxiv.org/abs/2102.09672). Este planificador es simétrico y evoluciona despacio al principio y final del proceso de difusión (cuando t es cercano a 0 o 1). Esto puede sonar complicado pero en realidad es sencillo (ver el código abajo): lo único que se hace es convertir el instante temporal entre 0 y 1 en un ángulo entre $0$ y $\pi$ y luego devolver su seno y su coseno, con lo que se consigue la simetría y que sus cuadrados sumen uno.


### Entrenamiento

El entrenamiento (ver `train_step()` y `denoise()`) va como sigue:

- Muestreamos tiempos utilizando una distribución uniforme, para cada tiempo obtenemos los ratios de ruido y señal y generamos las imágenes ruidosas.

- Entrenamos el modelo para separar las imágenes ruidosas en sus componentes de señal y ruido. Normalmente se usa el MSE para el entrenamiento, pero para este dataset parece que el MAE funciona mejor.

- El modelo tiene dos U-Net, la principal que se va entrenando y otra, `ema_network`, cuyos pesos se van actualizando más lentamente siguiendo una media móvil exponencial (*exponential moving average = ema*), y que **es la red que se usa en inferencia**. Es decir los pesos de esta red se ajustan como $w_{ema}=\beta * w_{ema} + (1-\beta) * w$, donde $w_{ema}$ son los pesos de la `ema_network`, $w$ los pesos de la red principal, y $\beta$ es una variable que toma un valor cercano a 1 (cuanto más cercano a 1, más lentamente se ajusta esta red). Eso se hace para que el entrenamiento de esta red ema, que es la que al final se va a usar para generar imágenes, sea más suave y estable.

### Muestreo (difusión inversa)

Al muestrear (ver `reverse_diffusion()`), en cada paso se toma la estimación previa de la imagen ruidosa y la separamos en sus componentes de imagen y ruido con nuestra red. Luego recombinamos estos componentes utilizando los ratios de ruido y señal del siguiente paso.

Para generar imágenes nuevas (ver `generate()`) una vez entrenada la red simplemente se genera ruido Gaussiano y se llama a `reverse_diffusion()`.

### Cosas relevantes del código

- En el modelo hay dos métricas, `n_loss` y `i_loss`. El primero se usa para evaluar cómo de bien estima la red el ruido que se le ha añadido a la imagen. La red estima dicho ruido usando sólo la imagen ruidosa y el ratio de ruido, codificado previavemente usando el *sinusoidal embedding*.

- Una vez estimado el ruido se obtiene la imagen limpia básicamente restando el ruido estimado. La calidad de la imagen limpia se mide con el `i_loss`.

- Lo importante es que para entrenar el model **solo se usa `n_loss`**, la `i_loss` solo se usa como métrica.

In [ ]:
@keras.saving.register_keras_serializable()
class DiffusionModel(keras.Model):
    def __init__(self, image_size, widths, block_depth):
        super().__init__()

        self.normalizer = layers.Normalization()
        self.network = get_network(image_size, widths, block_depth)
        self.ema_network = keras.models.clone_model(self.network)

    def compile(self, **kwargs):
        super().compile(**kwargs)

        self.noise_loss_tracker = keras.metrics.Mean(name="n_loss")
        self.image_loss_tracker = keras.metrics.Mean(name="i_loss")

    @property
    def metrics(self):
        return [self.noise_loss_tracker, self.image_loss_tracker]

    def denormalize(self, images):
        # convert the pixel values back to 0-1 range
        images = self.normalizer.mean + images * self.normalizer.variance**0.5
        return ops.clip(images, 0.0, 1.0)

    def diffusion_schedule(self, diffusion_times):
        # diffusion times -> angles
        start_angle = ops.cast(ops.arccos(max_signal_rate), "float32")
        end_angle = ops.cast(ops.arccos(min_signal_rate), "float32")

        diffusion_angles = start_angle + diffusion_times * (end_angle - start_angle)

        # angles -> signal and noise rates
        signal_rates = ops.cos(diffusion_angles)
        noise_rates = ops.sin(diffusion_angles)
        # note that their squared sum is always: sin^2(x) + cos^2(x) = 1

        return noise_rates, signal_rates

    def denoise(self, noisy_images, noise_rates, signal_rates, training):
        # the exponential moving average weights are used at evaluation
        if training:
            network = self.network
        else:
            network = self.ema_network

        # predict noise component and calculate the image component using it
        pred_noises = network([noisy_images, noise_rates**2], training=training)
        pred_images = (noisy_images - noise_rates * pred_noises) / signal_rates

        return pred_noises, pred_images

    def reverse_diffusion(self, initial_noise, diffusion_steps):
        # reverse diffusion = sampling
        num_images = initial_noise.shape[0]
        step_size = 1.0 / diffusion_steps

        # important line:
        # at the first sampling step, the "noisy image" is pure noise
        # but its signal rate is assumed to be nonzero (min_signal_rate)
        next_noisy_images = initial_noise
        for step in range(diffusion_steps):
            noisy_images = next_noisy_images

            # separate the current noisy image to its components
            diffusion_times = ops.ones((num_images, 1, 1, 1)) - step * step_size
            noise_rates, signal_rates = self.diffusion_schedule(diffusion_times)
            pred_noises, pred_images = self.denoise(
                noisy_images, noise_rates, signal_rates, training=False
            )
            # network used in eval mode

            # remix the predicted components using the next signal and noise rates
            next_diffusion_times = diffusion_times - step_size
            next_noise_rates, next_signal_rates = self.diffusion_schedule(
                next_diffusion_times
            )
            next_noisy_images = (
                next_signal_rates * pred_images + next_noise_rates * pred_noises
            )
            # this new noisy image will be used in the next step

        return pred_images

    def generate(self, num_images, diffusion_steps):
        # noise -> images -> denormalized images
        initial_noise = keras.random.normal(
            shape=(num_images, image_size, image_size, 3)
        )
        generated_images = self.reverse_diffusion(initial_noise, diffusion_steps)
        generated_images = self.denormalize(generated_images)
        return generated_images

    def train_step(self, images):
        # normalize images to have standard deviation of 1, like the noises
        images = self.normalizer(images, training=True)
        noises = keras.random.normal(shape=(batch_size, image_size, image_size, 3))

        # sample uniform random diffusion times
        diffusion_times = keras.random.uniform(
            shape=(batch_size, 1, 1, 1), minval=0.0, maxval=1.0
        )
        noise_rates, signal_rates = self.diffusion_schedule(diffusion_times)
        # mix the images with noises accordingly
        noisy_images = signal_rates * images + noise_rates * noises

        with tf.GradientTape() as tape:
            # train the network to separate noisy images to their components
            pred_noises, pred_images = self.denoise(
                noisy_images, noise_rates, signal_rates, training=True
            )

            noise_loss = self.loss(noises, pred_noises)  # used for training
            image_loss = self.loss(images, pred_images)  # only used as metric

        gradients = tape.gradient(noise_loss, self.network.trainable_weights)
        self.optimizer.apply_gradients(zip(gradients, self.network.trainable_weights))

        self.noise_loss_tracker.update_state(noise_loss)
        self.image_loss_tracker.update_state(image_loss)

        # track the exponential moving averages of weights
        for weight, ema_weight in zip(self.network.weights, self.ema_network.weights):
            ema_weight.assign(ema * ema_weight + (1 - ema) * weight)

        # Return metrics
        return {m.name: m.result() for m in self.metrics}

    def test_step(self, images):
        # normalize images to have standard deviation of 1, like the noises
        images = self.normalizer(images, training=False)
        noises = keras.random.normal(shape=(batch_size, image_size, image_size, 3))

        # sample uniform random diffusion times
        diffusion_times = keras.random.uniform(
            shape=(batch_size, 1, 1, 1), minval=0.0, maxval=1.0
        )
        noise_rates, signal_rates = self.diffusion_schedule(diffusion_times)
        # mix the images with noises accordingly
        noisy_images = signal_rates * images + noise_rates * noises

        # use the network to separate noisy images to their components
        pred_noises, pred_images = self.denoise(
            noisy_images, noise_rates, signal_rates, training=False
        )

        noise_loss = self.loss(noises, pred_noises)
        image_loss = self.loss(images, pred_images)

        self.image_loss_tracker.update_state(image_loss)
        self.noise_loss_tracker.update_state(noise_loss)

        # measure KID between real and generated images
        # this is computationally demanding, kid_diffusion_steps has to be small
        # images = self.denormalize(images)
        # generated_images = self.generate(
        #     num_images=batch_size, diffusion_steps=kid_diffusion_steps
        # )
        # self.kid.update_state(images, generated_images)

        return {m.name: m.result() for m in self.metrics}

    def plot_images(self, epoch=None, logs=None, num_rows=3, num_cols=6):
        # plot random generated images for visual evaluation of generation quality
        generated_images = self.generate(
            num_images=num_rows * num_cols,
            diffusion_steps=plot_diffusion_steps,
        )

        plt.figure(figsize=(num_cols * 2.0, num_rows * 2.0))
        for row in range(num_rows):
            for col in range(num_cols):
                index = row * num_cols + col
                plt.subplot(num_rows, num_cols, index + 1)
                plt.imshow(generated_images[index])
                plt.axis("off")
        plt.tight_layout()
        plt.show()
        plt.close()

## Training

In [ ]:
# create and compile the model
model = DiffusionModel(image_size, widths, block_depth)
# below tensorflow 2.9:
# pip install tensorflow_addons
# import tensorflow_addons as tfa
# optimizer=tfa.optimizers.AdamW
model.compile(
    optimizer=keras.optimizers.AdamW(
        learning_rate=learning_rate, weight_decay=weight_decay
    ),
    loss=keras.losses.mean_absolute_error,
)
# pixelwise mean absolute error is used as loss

# calculate mean and variance of training dataset for normalization
model.normalizer.adapt(train_dataset)

In [ ]:
# If we want to load the model weights => it works for inference, maybe not for training
model.build((image_size, image_size, 3))
model.load_weights("dm_model_10_epochs.keras")
# Another option is to run model.fit with just one epoch and then use model.load_weights

In [ ]:
num_epochs = 10

# run training and plot generated images periodically
hist = model.fit(
    train_dataset,
    epochs=num_epochs,
    validation_data=val_dataset,
    callbacks=[
        keras.callbacks.LambdaCallback(on_epoch_end=model.plot_images),
    ],
)

## Inferencia (generar nuevas imágenes)

In [ ]:
# generate images
model.plot_images()

In [ ]:
# Not tested, but should work
# model.build((image_size, image_size, 3))
# model.save_weights('dm_model_10_epochs.weights.h5')

# Tested, this works
# model.save('dm_model_10_epochs.keras')

## Resultados

Con 50 épocas (2 horas en una GPU T4 como las del colab) salen imágenes de calidad.

Aquí se ve como evoluciona el entrenamiento en 80 épocas:

![flowers training gif](https://i.imgur.com/FSCKtZq.gif)

Imágenes generadas usando entre 1 y 20 pasos de muestreo partiendo desde el mismo ruido:

![flowers sampling steps gif](https://i.imgur.com/tM5LyH3.gif)

Interpolación esférica entre el ruido inicial:

![flowers interpolation gif](https://i.imgur.com/hk5Hd5o.gif)

En las dos siguientes se compara el usar muestreo determinista o estocástico.

Muestreo determinista, imágenes ruidosas arriba y predichas abajo, 40 pasos:

![flowers deterministic generation gif](https://i.imgur.com/wCvzynh.gif)

Muestreo estocástico, imágenes ruidosas arriba y predichas abajo, 80 pasos:

![flowers stochastic generation gif](https://i.imgur.com/kRXOGzd.gif)

## Conclusiones

Echar un vistazo a [este repo](https://github.com/beresandras/clear-diffusion-keras) que contiene más experimentos.

### Consejos

* **min. and max. signal rates**: El min. signal rate es importante, si es muy bajo las imágenes salen sobresaturadas, si es muy alto al revés. Si se pone a 0 se produce un error de división por cero. El max. signal rate se puede poner a 1 pero con un valor un poco menor genera mejores imágenes.

* **loss function**: Normalmente se usa MSE, que genera imágenes más diversas, pero "alucina" más ([ver aquí Section 3](https://arxiv.org/abs/2111.05826)). MAE genera imágenes más "suaves". Lo mejor es probar con ambos y comparar.

* **weight decay**: Este parámetro del optimizador AdamW hace más estable el entrenamiento.

* **exponential moving average of weights**: Ayuda al obtener un entrenamiento más suave al promediar.

* **data normalization**: Normalmente se escalan las imágenes entre -1 y 1, aunque aquí tiene más sentido normalizar para obtener imágenes con media 0 y varianza 1, igual que los ruidos añadidos.

* **noise level input**: Hay varias opciones que se pueden probar, usar la varianza del ruido (en este ejemplo), el ratio de ruido (sale parecido), el ratio de señal (sale peor) o la relación señal ruido.

* **learning rate**: En este ejemplo y con el Adam 1e-3 va bien, pero en general se usan valores más pequeños.

* **sinusoidal embedding**: Esto es crucial para obtener un buen rendimiento. El autor recomienda usar como frecuencia mínima la inversa del rango de la entrada. En este caso como se usa la varianza del ruido se puede dejar a 1. La frecuencia máxima controla el valor más pequeño de la varianza del ruido al cual la red es sensible, y las dimensiones (`embedding_dims`) el número de componentes frecuenciales. Estos valores no son críticos para el rendimiento.

* **skip connections**: Usar skip connections es fundamental para que el modelo funcione, si no no aprende a limpiar bien las imágenes.

* **residual connections**: También son muy importantes, en este caso porque la información del nivel de ruido solo se mete a la entrada en lugar de a las capas intermedias.

* **normalization**: Importante. Se pueden usar varios tipos pero con *BacthNormalization* funciona bien.

* **activations**: Tienen bastante impacto en la calidad de las imágenes generadas. Las funciones no monotónicas van mejor que las monotónicas, como la ReLU. La mejor para este ejemplo es la [Swish](https://www.tensorflow.org/api_docs/python/tf/keras/activations/swish).

* **attention**: En los modelos de difusión se suelen usar capas de atención para mejorar la coherencia global. Por simplicidad no se han usado aquí.

* **upsampling**: Se puede usar *bilinear* o *nearest neighbour* para obtener resultados parecidos. Otra opción que se podría probar es usar [convoluciones transpuestas](https://keras.io/api/layers/convolution_layers/convolution2d_transpose/).

## Ejercicios
- Adapta el código para que funcione con MNIST y genere dígitos.

In [ ]:
# Ejercicio: Adaptar el modelo DDIM para MNIST (digitos en escala de grises)
#
# Cambios principales respecto al codigo original de flores:
# 1. Dataset: MNIST en vez de Oxford Flowers 102
# 2. image_size: 28x28 (mantenemos la resolucion original de MNIST)
# 3. Canales: 1 (escala de grises) en vez de 3 (RGB)
# 4. Arquitectura mas pequena (widths mas estrechos) porque las imagenes son mas simples
# 5. Normalizacion adaptada: MNIST ya esta en [0,1] tras /255

import math
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers
from tensorflow.keras import ops

# --- Hiperparametros para MNIST ---
image_size_mnist    = 28
num_channels_mnist  = 1       # escala de grises
plot_diffusion_steps_mnist = 20

# sampling
min_signal_rate_m = 0.02
max_signal_rate_m = 0.95

# architecture (mas estrecha que la de flores)
embedding_dims_m        = 32
embedding_max_frequency_m = 1000.0
widths_m    = [16, 32, 64]   # 3 niveles bastan para imagenes 28x28
block_depth_m = 2

# optimization
batch_size_m  = 64
ema_m         = 0.999
learning_rate_m = 1e-3
weight_decay_m  = 1e-4

# --- Pipeline de datos MNIST ---
(X_mnist_train, _), (X_mnist_test, _) = tf.keras.datasets.mnist.load_data()
# Combinar train + test para tener mas datos y repetir
X_all_mnist = np.concatenate([X_mnist_train, X_mnist_test], axis=0)  # (70000, 28, 28)
X_all_mnist = X_all_mnist.astype(np.float32) / 255.0                 # [0, 1]
X_all_mnist = np.expand_dims(X_all_mnist, -1)                        # (70000, 28, 28, 1)

def prepare_mnist_dataset(data, val_split=0.2):
    n_val = int(len(data) * val_split)
    train_data = data[n_val:]
    val_data   = data[:n_val]
    def make_ds(arr):
        return (tf.data.Dataset.from_tensor_slices(arr)
                .cache()
                .repeat(5)
                .shuffle(10 * batch_size_m)
                .batch(batch_size_m, drop_remainder=True)
                .prefetch(tf.data.AUTOTUNE))
    return make_ds(train_data), make_ds(val_data)

train_dataset_m, val_dataset_m = prepare_mnist_dataset(X_all_mnist)
print('Dataset MNIST listo. Ejemplo batch shape:', next(iter(train_dataset_m)).shape)

# --- Red U-Net adaptada a 1 canal ---
@keras.saving.register_keras_serializable()
def sinusoidal_embedding_m(x):
    embedding_min_frequency_m = 1.0
    frequencies = ops.exp(
        ops.linspace(
            ops.log(embedding_min_frequency_m),
            ops.log(embedding_max_frequency_m),
            embedding_dims_m // 2,
        )
    )
    angular_speeds = ops.cast(2.0 * math.pi * frequencies, 'float32')
    embeddings = ops.concatenate(
        [ops.sin(angular_speeds * x), ops.cos(angular_speeds * x)], axis=3
    )
    return embeddings

def ResidualBlock_m(width):
    def apply(x):
        input_width = x.shape[3]
        residual = x if input_width == width else layers.Conv2D(width, kernel_size=1)(x)
        x = layers.BatchNormalization(center=False, scale=False)(x)
        x = layers.Conv2D(width, kernel_size=3, padding='same', activation='swish')(x)
        x = layers.Conv2D(width, kernel_size=3, padding='same')(x)
        return layers.Add()([x, residual])
    return apply

def DownBlock_m(width, block_depth):
    def apply(x):
        x, skips = x
        for _ in range(block_depth):
            x = ResidualBlock_m(width)(x); skips.append(x)
        x = layers.AveragePooling2D(pool_size=2)(x)
        return x
    return apply

def UpBlock_m(width, block_depth):
    def apply(x):
        x, skips = x
        x = layers.UpSampling2D(size=2, interpolation='bilinear')(x)
        for _ in range(block_depth):
            x = layers.Concatenate()([x, skips.pop()]); x = ResidualBlock_m(width)(x)
        return x
    return apply

def get_network_mnist(image_size, widths, block_depth, num_channels=1):
    noisy_images   = keras.Input(shape=(image_size, image_size, num_channels))
    noise_variances = keras.Input(shape=(1, 1, 1))

    e = layers.Lambda(sinusoidal_embedding_m,
                      output_shape=(1, 1, embedding_dims_m))(noise_variances)
    e = layers.UpSampling2D(size=image_size, interpolation='nearest')(e)

    x = layers.Conv2D(widths[0], kernel_size=1)(noisy_images)
    x = layers.Concatenate()([x, e])

    skips = []
    for width in widths[:-1]:
        x = DownBlock_m(width, block_depth)([x, skips])
    for _ in range(block_depth):
        x = ResidualBlock_m(widths[-1])(x)
    for width in reversed(widths[:-1]):
        x = UpBlock_m(width, block_depth)([x, skips])

    # Salida con num_channels canales (1 para MNIST)
    x = layers.Conv2D(num_channels, kernel_size=1, kernel_initializer='zeros')(x)
    return keras.Model([noisy_images, noise_variances], x, name='unet_mnist')

# --- Modelo de difusion adaptado ---
@keras.saving.register_keras_serializable()
class DiffusionModelMNIST(keras.Model):
    def __init__(self, image_size, widths, block_depth, num_channels=1):
        super().__init__()
        self.num_channels = num_channels
        self.image_size   = image_size
        self.normalizer   = layers.Normalization()
        self.network      = get_network_mnist(image_size, widths, block_depth, num_channels)
        self.ema_network  = keras.models.clone_model(self.network)

    def compile(self, **kwargs):
        super().compile(**kwargs)
        self.noise_loss_tracker = keras.metrics.Mean(name='n_loss')
        self.image_loss_tracker = keras.metrics.Mean(name='i_loss')

    @property
    def metrics(self):
        return [self.noise_loss_tracker, self.image_loss_tracker]

    def denormalize(self, images):
        images = self.normalizer.mean + images * self.normalizer.variance**0.5
        return ops.clip(images, 0.0, 1.0)

    def diffusion_schedule(self, diffusion_times):
        start_angle = ops.cast(ops.arccos(max_signal_rate_m), 'float32')
        end_angle   = ops.cast(ops.arccos(min_signal_rate_m), 'float32')
        angles      = start_angle + diffusion_times * (end_angle - start_angle)
        return ops.sin(angles), ops.cos(angles)  # noise_rate, signal_rate

    def denoise(self, noisy_images, noise_rates, signal_rates, training):
        network = self.network if training else self.ema_network
        pred_noises = network([noisy_images, noise_rates**2], training=training)
        pred_images = (noisy_images - noise_rates * pred_noises) / signal_rates
        return pred_noises, pred_images

    def reverse_diffusion(self, initial_noise, diffusion_steps):
        num_images = initial_noise.shape[0]
        step_size  = 1.0 / diffusion_steps
        next_noisy_images = initial_noise
        for step in range(diffusion_steps):
            noisy_images   = next_noisy_images
            diffusion_times = ops.ones((num_images, 1, 1, 1)) - step * step_size
            noise_rates, signal_rates = self.diffusion_schedule(diffusion_times)
            pred_noises, pred_images  = self.denoise(
                noisy_images, noise_rates, signal_rates, training=False)
            next_diffusion_times = diffusion_times - step_size
            next_noise_rates, next_signal_rates = self.diffusion_schedule(next_diffusion_times)
            next_noisy_images = (next_signal_rates * pred_images +
                                 next_noise_rates  * pred_noises)
        return pred_images

    def generate(self, num_images, diffusion_steps):
        initial_noise   = keras.random.normal(
            shape=(num_images, self.image_size, self.image_size, self.num_channels))
        generated_images = self.reverse_diffusion(initial_noise, diffusion_steps)
        return self.denormalize(generated_images)

    def train_step(self, images):
        images = self.normalizer(images, training=True)
        noises = keras.random.normal(
            shape=(batch_size_m, self.image_size, self.image_size, self.num_channels))
        diffusion_times = keras.random.uniform(
            shape=(batch_size_m, 1, 1, 1), minval=0.0, maxval=1.0)
        noise_rates, signal_rates = self.diffusion_schedule(diffusion_times)
        noisy_images = signal_rates * images + noise_rates * noises
        with tf.GradientTape() as tape:
            pred_noises, pred_images = self.denoise(
                noisy_images, noise_rates, signal_rates, training=True)
            noise_loss = self.loss(noises, pred_noises)
            image_loss = self.loss(images, pred_images)
        gradients = tape.gradient(noise_loss, self.network.trainable_weights)
        self.optimizer.apply_gradients(zip(gradients, self.network.trainable_weights))
        self.noise_loss_tracker.update_state(noise_loss)
        self.image_loss_tracker.update_state(image_loss)
        for w, ew in zip(self.network.weights, self.ema_network.weights):
            ew.assign(ema_m * ew + (1 - ema_m) * w)
        return {m.name: m.result() for m in self.metrics}

    def test_step(self, images):
        images = self.normalizer(images, training=False)
        noises = keras.random.normal(
            shape=(batch_size_m, self.image_size, self.image_size, self.num_channels))
        diffusion_times = keras.random.uniform(
            shape=(batch_size_m, 1, 1, 1), minval=0.0, maxval=1.0)
        noise_rates, signal_rates = self.diffusion_schedule(diffusion_times)
        noisy_images = signal_rates * images + noise_rates * noises
        pred_noises, pred_images = self.denoise(
            noisy_images, noise_rates, signal_rates, training=False)
        self.noise_loss_tracker.update_state(self.loss(noises, pred_noises))
        self.image_loss_tracker.update_state(self.loss(images, pred_images))
        return {m.name: m.result() for m in self.metrics}

    def plot_images(self, epoch=None, logs=None, num_rows=3, num_cols=6):
        generated = self.generate(
            num_images=num_rows * num_cols,
            diffusion_steps=plot_diffusion_steps_mnist)
        plt.figure(figsize=(num_cols * 2.0, num_rows * 2.0))
        for row in range(num_rows):
            for col in range(num_cols):
                idx = row * num_cols + col
                plt.subplot(num_rows, num_cols, idx + 1)
                plt.imshow(generated[idx, :, :, 0], cmap='gray')
                plt.axis('off')
        plt.tight_layout(); plt.show(); plt.close()

# --- Compilar y entrenar ---
model_mnist = DiffusionModelMNIST(image_size_mnist, widths_m, block_depth_m,
                                   num_channels=num_channels_mnist)
model_mnist.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=learning_rate_m,
                                     weight_decay=weight_decay_m),
    loss=keras.losses.mean_absolute_error,
)
model_mnist.normalizer.adapt(train_dataset_m)

num_epochs_mnist = 10  # aumentar a 50+ para mejores resultados
hist_mnist = model_mnist.fit(
    train_dataset_m,
    epochs=num_epochs_mnist,
    validation_data=val_dataset_m,
    callbacks=[keras.callbacks.LambdaCallback(on_epoch_end=model_mnist.plot_images)],
)

# --- Generar digitos al final ---
model_mnist.plot_images()

# Mostrar curvas de entrenamiento
plt.plot(hist_mnist.history['n_loss'], label='n_loss train')
plt.plot(hist_mnist.history['val_n_loss'], label='n_loss val')
plt.legend(); plt.title('DDIM MNIST - Curvas de entrenamiento'); plt.show()


## What to try next?

If you would like to dive in deeper to the topic, I recommend checking out
[this repository](https://github.com/beresandras/clear-diffusion-keras) that I created in
preparation for this code example, which implements a wider range of features in a
similar style, such as:

* stochastic sampling
* second-order sampling based on the
[differential equation view of DDIMs (Equation 13)](https://arxiv.org/abs/2010.02502)
* more diffusion schedules
* more network output types: predicting image or
[velocity (Appendix D)](https://arxiv.org/abs/2202.00512) instead of noise
* more datasets

## Related works

* [Score-based generative modeling](https://yang-song.github.io/blog/2021/score/)
(blogpost)
* [What are diffusion models?](https://lilianweng.github.io/posts/2021-07-11-diffusion-models/)
(blogpost)
* [Annotated diffusion model](https://huggingface.co/blog/annotated-diffusion) (blogpost)
* [CVPR 2022 tutorial on diffusion models](https://cvpr2022-tutorial-diffusion-models.github.io/)
(slides available)
* [Elucidating the Design Space of Diffusion-Based Generative Models](https://arxiv.org/abs/2206.00364):
attempts unifying diffusion methods under a common framework
* High-level video overviews: [1](https://www.youtube.com/watch?v=yTAMrHVG1ew),
[2](https://www.youtube.com/watch?v=344w5h24-h8)
* Detailed technical videos: [1](https://www.youtube.com/watch?v=fbLgFrlTnGU),
[2](https://www.youtube.com/watch?v=W-O7AZNzbzQ)
* Score-based generative models: [NCSN](https://arxiv.org/abs/1907.05600),
[NCSN+](https://arxiv.org/abs/2006.09011), [NCSN++](https://arxiv.org/abs/2011.13456)
* Denoising diffusion models: [DDPM](https://arxiv.org/abs/2006.11239),
[DDIM](https://arxiv.org/abs/2010.02502), [DDPM+](https://arxiv.org/abs/2102.09672),
[DDPM++](https://arxiv.org/abs/2105.05233)
* Large diffusion models: [GLIDE](https://arxiv.org/abs/2112.10741),
[DALL-E 2](https://arxiv.org/abs/2204.06125/), [Imagen](https://arxiv.org/abs/2205.11487)
